# Day 02 - Lazy Evaluation

**Topic:** Lazy Evaluation  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจว่า Spark จะยังไม่ execute transformations ทันที จนกว่าจะเจอ action

วันนี้เราจะเรียนให้เห็นภาพว่า PySpark DataFrame code ที่เขียนต่อกันหลาย step ไม่ได้แปลว่า Spark ทำงานทันทีทุกบรรทัด แต่ Spark จะสร้าง execution plan ไว้ก่อน แล้วค่อย execute จริงตอนเจอ action เช่น `.show()`, `.count()`, `.write`, `.collect()`

## Goal of the day

1. แยกให้ออกว่า operation ไหนคือ **Transformation** และ operation ไหนคือ **Action**
2. อธิบายได้ว่า **Lazy Evaluation** คืออะไรในมุมของ Spark
3. ใช้ `.explain()` เพื่อดู execution plan เบื้องต้นได้
4. เข้าใจว่าทำไม `.show()`, `.count()`, `.write` ถึง trigger Spark job
5. เริ่มระวัง repeated actions ที่ทำให้ Spark compute ซ้ำโดยไม่จำเป็น

## Why it matters in production

ใน production pipeline เรื่อง Lazy Evaluation สำคัญมาก เพราะ:

- Spark จะยังไม่ run transformation จริงจนกว่าจะเจอ action
- การเรียก `.count()`, `.show()`, `.collect()` หลายครั้ง อาจทำให้ Spark compute ซ้ำหลายรอบ
- ถ้า pipeline มี join / aggregation / filter หลาย step Spark จะ optimize plan ก่อน execute
- การอ่าน `.explain()` ช่วยให้เริ่ม debug ได้ว่า Spark จะทำอะไร เช่น filter, projection, aggregation
- ช่วยป้องกัน mindset แบบ pandas ที่คิดว่า code ทุกบรรทัด execute ทันที

Production takeaway:

> Spark code is declarative. เราเขียนว่าอยากได้ผลลัพธ์อะไร จากนั้น Spark จะวางแผนและ execute ตอนเจอ action

## Key concepts

| Concept | Meaning |
|---|---|
| Lazy Evaluation | Spark ยังไม่ execute ทันที แต่เก็บ plan ไว้ก่อน |
| Transformation | Operation ที่สร้าง DataFrame ใหม่ เช่น `select`, `filter`, `withColumn`, `groupBy` |
| Action | Operation ที่ trigger execution เช่น `show`, `count`, `collect`, `write` |
| Logical Plan | แผนเชิง logic ว่าต้องทำ transformation อะไร |
| Physical Plan | แผนจริงที่ Spark จะใช้ execute |
| Job | งานที่ Spark submit เมื่อมี action |
| Stage | ส่วนของ job ที่ถูกแบ่งตาม shuffle boundary |
| Task | งานย่อยที่ executor ทำกับ partition |

## Concept explanation

### Lazy Evaluation คืออะไร?

ใน PySpark เวลาเราเขียน code เช่น:

```python
df_filtered = df.filter("amount > 1000")
df_selected = df_filtered.select("transaction_id", "amount")
```

Spark ยังไม่ได้ scan data จริงทันที แต่จะจำไว้ว่า:

1. ต้อง filter records ที่ `amount > 1000`
2. ต้องเลือกเฉพาะ columns ที่กำหนด
3. ถ้ามี action ค่อย execute plan นี้

พูดง่าย ๆ:

> Transformation = ต่อแผน  
> Action = สั่งให้ Spark ลงมือทำจริง

### Transformation vs Action

Transformation มักจะ return DataFrame ใหม่ เช่น:

- `select`
- `filter`
- `where`
- `withColumn`
- `drop`
- `join`
- `groupBy().agg()`

Action มักจะ return result กลับมาหา driver หรือเขียน output เช่น:

- `show`
- `count`
- `collect`
- `take`
- `first`
- `write`
- `saveAsTable`

### ทำไม Spark ต้อง lazy?

เพราะ Spark สามารถ optimize plan ก่อน run จริง เช่น:

- ตัด column ที่ไม่จำเป็นออก (**projection pruning**)
- ดัน filter ให้ทำเร็วขึ้น (**predicate pushdown** ในบาง source)
- รวม transformation หลาย step เป็น plan เดียว
- เลือก physical strategy ที่เหมาะสมกว่า เช่น
  - วิธี join เช่น Broadcast Join หรือ Sort-Merge Join
  - การ shuffle data เมื่อจำเป็น
  - การ scan เฉพาะ columns หรือ partitions ที่เกี่ยวข้อง
  - ลำดับการ execute ที่ลด data movement ได้มากขึ้น

ถ้า Spark execute ทุกบรรทัดทันที จะ optimize ภาพรวมได้น้อยลงและอาจเสีย performance มากกว่า


## Mermaid diagram: Lazy Evaluation Flow

```mermaid
flowchart LR
    A[Create DataFrame] --> B[Transformation: filter]
    B --> C[Transformation: withColumn]
    C --> D[Transformation: select]
    D --> E{Action?}
    E -- No --> F[Only build logical plan]
    E -- Yes: show/count/write --> G[Spark creates physical plan]
    G --> H[Submit Spark Job]
    H --> I[Stages and Tasks run on executors]
    I --> J[Return result or write output]
```

Key idea:

> ก่อนถึง action ส่วนใหญ่ Spark แค่ build plan ยังไม่ได้ process data จริงแบบเต็มรูปแบบ

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 3, Finished, Available, Finished, False)

## Create mock data

In [2]:
transactions_data = [
    ("T001", 101, "Bangkok", 1200.50, "success", "2026-05-01"),
    ("T002", 102, "Bangkok", 250.00, "success", "2026-05-01"),
    ("T003", 103, "Chiang Mai", 3400.00, "success", "2026-05-02"),
    ("T004", 104, "Rayong", 0.00, "failed", "2026-05-02"),
    ("T005", 105, "Bangkok", 780.00, "pending", "2026-05-03"),
    ("T006", 106, "Phuket", 5600.00, "success", "2026-05-03"),
    ("T007", 107, "Bangkok", 150.00, "failed", "2026-05-04"),
    ("T008", 108, "Chiang Mai", 980.00, "success", "2026-05-04"),
]

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), True),
])

df_transactions = spark.createDataFrame(transactions_data, transaction_schema)

df_transactions.show()
df_transactions.printSchema()

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 4, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-------+----------------+
|transaction_id|customer_id|    region|amount| status|transaction_date|
+--------------+-----------+----------+------+-------+----------------+
|          T001|        101|   Bangkok|1200.5|success|      2026-05-01|
|          T002|        102|   Bangkok| 250.0|success|      2026-05-01|
|          T003|        103|Chiang Mai|3400.0|success|      2026-05-02|
|          T004|        104|    Rayong|   0.0| failed|      2026-05-02|
|          T005|        105|   Bangkok| 780.0|pending|      2026-05-03|
|          T006|        106|    Phuket|5600.0|success|      2026-05-03|
|          T007|        107|   Bangkok| 150.0| failed|      2026-05-04|
|          T008|        108|Chiang Mai| 980.0|success|      2026-05-04|
+--------------+-----------+----------+------+-------+----------------+

root
 |-- transaction_id: string (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- region: string (nullable = true)


## PySpark code examples

ใน section นี้จะไล่จาก transformation chain → explain plan → action → write table → cache เพื่อให้เห็นภาพ execution behavior แบบครบ flow

### Example 1: Transformation chain does not execute immediately

Cell นี้สร้าง DataFrame ใหม่จากหลาย transformation:

- `filter`
- `withColumn`
- `select`

แต่จนกว่าจะเรียก action เช่น `.show()` หรือ `.count()` Spark ยังไม่ได้ประมวลผลข้อมูลเต็ม ๆ จริง

In [3]:
df_high_value_success = (
    df_transactions
    .filter(F.col("status") == "success")
    .filter(F.col("amount") >= 1000)
    .withColumn("amount_band", F.lit("high_value"))  # F.lit() adds a constant value to every row
    .select(
        "transaction_id",
        "customer_id",
        "region",
        "amount",
        "amount_band",
        "transaction_date"
    )
)

print("DataFrame transformation chain has been defined.")
print("No action has been called yet, so Spark has mainly built a plan.")

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 5, Finished, Available, Finished, False)

DataFrame transformation chain has been defined.
No action has been called yet, so Spark has mainly built a plan.


### Example 2: Use explain to inspect the plan

`.explain()` ใช้ดู plan ว่า Spark เข้าใจ transformation chain นี้อย่างไร

สำคัญ: `.explain()` ช่วย inspect plan แต่ไม่ได้ return data result เหมือน `.show()` หรือ `.count()`

In [4]:
df_high_value_success.explain(mode="formatted")

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 6, Finished, Available, Finished, False)

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]
Arguments: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]
Condition : ((isnotnull(status#730) AND isnotnull(amount#729)) AND ((status#730 = success) AND (amount#729 >= 1000.0)))

(3) Project [codegen id : 1]
Output [6]: [transaction_id#726, customer_id#727, region#728, amount#729, high_value AS amount_band#763, transaction_date#731]
Input [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]




### Example 3: Action triggers execution

เมื่อเรียก `.show()` Spark จะเริ่ม execute plan จริง และแสดงผลลัพธ์กลับมาที่ notebook

In [5]:
df_high_value_success.show()

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 7, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-----------+----------------+
|transaction_id|customer_id|    region|amount|amount_band|transaction_date|
+--------------+-----------+----------+------+-----------+----------------+
|          T001|        101|   Bangkok|1200.5| high_value|      2026-05-01|
|          T003|        103|Chiang Mai|3400.0| high_value|      2026-05-02|
|          T006|        106|    Phuket|5600.0| high_value|      2026-05-03|
+--------------+-----------+----------+------+-----------+----------------+



### Example 4: Another action can trigger another execution

ถ้าเรียก `.count()` หลังจาก `.show()` Spark อาจต้อง execute plan อีกครั้ง เพราะ DataFrame ไม่ได้ถูก cache ไว้

สำหรับ dataset เล็ก ๆ ไม่เป็นไร แต่ใน production dataset ใหญ่ ๆ การ action ซ้ำโดยไม่จำเป็นอาจแพงมาก

In [6]:
high_value_count = df_high_value_success.count()
print("High value successful transaction count:", high_value_count)

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 8, Finished, Available, Finished, False)

High value successful transaction count: 3


### Example 5: Aggregation is also lazy until action

`groupBy().agg()` เป็น transformation ที่สร้าง aggregated DataFrame ใหม่ แต่ยังไม่ run จริงจนกว่าจะเจอ action

In [7]:
df_region_summary = (
    df_transactions
    .filter(F.col("status") == "success")
    .groupBy("region")
    .agg(
        F.count("*").alias("success_transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_success_amount"),
        F.round(F.avg("amount"), 2).alias("avg_success_amount")
    )
    .orderBy(F.desc("total_success_amount"))
)

print("Aggregation plan has been defined.")

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 9, Finished, Available, Finished, False)

Aggregation plan has been defined.


ดู physical plan ของ aggregation

In [8]:
df_region_summary.explain(mode="formatted")

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 10, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]
Arguments: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]
Condition : (isnotnull(status#730) AND (status#730 = success))

(3) Project
Output [2]: [region#728, amount#729]
Input [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]

(4) HashAggregate
Input [2]: [region#

Trigger action ด้วย `.show()`

In [9]:
df_region_summary.show()

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 11, Finished, Available, Finished, False)

+----------+-------------------------+--------------------+------------------+
|    region|success_transaction_count|total_success_amount|avg_success_amount|
+----------+-------------------------+--------------------+------------------+
|    Phuket|                        1|              5600.0|            5600.0|
|Chiang Mai|                        2|              4380.0|            2190.0|
|   Bangkok|                        2|              1450.5|            725.25|
+----------+-------------------------+--------------------+------------------+



### Example 6: Write is also an action

ใน Fabric Lakehouse, การ write table จะ trigger Spark job เช่นเดียวกับ `.show()` หรือ `.count()`

Cell ด้านล่างจะเขียนผล summary เป็น Lakehouse table ชื่อ `day02_region_summary`

> หมายเหตุ: ต้อง attach Lakehouse เข้ากับ Fabric Notebook ก่อนรัน cell นี้

In [10]:
table_name = "day02_region_summary"

(
    df_region_summary
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"Table written successfully: {table_name}")

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 12, Finished, Available, Finished, False)

Table written successfully: day02_region_summary


### Example 7: Read table back from Lakehouse

หลังจาก write table แล้ว สามารถ read กลับด้วย `spark.read.table()`

การอ่าน table สร้าง DataFrame ก่อน และจะ execute เมื่อเจอ action เช่น `.show()`

In [11]:
df_saved_summary = spark.read.table("day02_region_summary")

print("Table DataFrame has been created.")
df_saved_summary.show()

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 13, Finished, Available, Finished, False)

Table DataFrame has been created.
+----------+-------------------------+--------------------+------------------+
|    region|success_transaction_count|total_success_amount|avg_success_amount|
+----------+-------------------------+--------------------+------------------+
|    Phuket|                        1|              5600.0|            5600.0|
|Chiang Mai|                        2|              4380.0|            2190.0|
|   Bangkok|                        2|              1450.5|            725.25|
+----------+-------------------------+--------------------+------------------+



### Optional: Cache to avoid repeated computation

ถ้า DataFrame เดิมถูกใช้ซ้ำหลาย action และ compute แพง อาจใช้ `.cache()` หรือ `.persist()`

แต่ cache ก็ใช้ memory ดังนั้นไม่ควร cache ทุกอย่างแบบสุ่ม ๆ

หลักคิดง่าย ๆ:

- ใช้ซ้ำหลายครั้งจริงไหม?
- DataFrame ใหญ่เกิน memory ไหม?
- คุ้มกับ memory ที่เสียไปไหม?
- หลังใช้เสร็จควร `unpersist()` ไหม?

In [12]:
df_cached = df_high_value_success.cache()

# First action materializes the cache.
df_cached.count()

# Second action can reuse cached result.
df_cached.show()

# Clean up cache when no longer needed.
df_cached.unpersist()

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 14, Finished, Available, Finished, False)

+--------------+-----------+----------+------+-----------+----------------+
|transaction_id|customer_id|    region|amount|amount_band|transaction_date|
+--------------+-----------+----------+------+-----------+----------------+
|          T001|        101|   Bangkok|1200.5| high_value|      2026-05-01|
|          T003|        103|Chiang Mai|3400.0| high_value|      2026-05-02|
|          T006|        106|    Phuket|5600.0| high_value|      2026-05-03|
+--------------+-----------+----------+------+-----------+----------------+



DataFrame[transaction_id: string, customer_id: int, region: string, amount: double, amount_band: string, transaction_date: string]

## Quick recap

| Question | Answer |
|---|---|
| Transformation execute ทันทีไหม? | ไม่ทันที Spark จะ build plan ไว้ก่อน |
| อะไรเป็นตัว trigger execution? | Action เช่น `show`, `count`, `collect`, `write` |
| ทำไม Spark ต้องใช้ Lazy Evaluation? | เพื่อให้ Spark optimize plan ก่อน run จริง |
| Repeated actions เสี่ยงอะไร? | อาจทำให้ Spark compute ซ้ำและใช้ resource เกินจำเป็น |
| ใช้อะไรดู execution plan ได้? | ใช้ `.explain()` หรือ `.explain(mode="formatted")` |

## Exercises

### Exercise 1: Build a lazy transformation chain

สร้าง DataFrame ชื่อ `df_bangkok_success` จาก `df_transactions`

Requirements:

- filter เฉพาะ `region = "Bangkok"`
- filter เฉพาะ `status = "success"`
- เพิ่ม column `amount_with_fee` = `amount * 1.03`
- select columns:
  - `transaction_id`
  - `customer_id`
  - `region`
  - `amount`
  - `amount_with_fee`

Expected result:

- ตอนสร้าง DataFrame ยังไม่ควรมี output table
- ใช้ `.explain()` เพื่อดู plan
- ใช้ `.show()` เพื่อ trigger execution

In [13]:
df_bangkok_success = (
    df_transactions
    .filter(F.col("region") == "Bangkok")
    .filter(F.col("status") == "success")
    .withColumn("amount_with_fee", F.round(F.col("amount") * F.lit(1.03), 2))
    .select(
        "transaction_id",
        "customer_id",
        "region",
        "amount",
        "amount_with_fee"
    )
)

df_bangkok_success.explain(mode="formatted")
df_bangkok_success.show()

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 15, Finished, Available, Finished, False)

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]
Arguments: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]
Condition : ((isnotnull(region#728) AND isnotnull(status#730)) AND ((region#728 = Bangkok) AND (status#730 = success)))

(3) Project [codegen id : 1]
Output [5]: [transaction_id#726, customer_id#727, region#728, amount#729, round((amount#729 * 1.03), 2) AS amount_with_fee#2250]
Input [6]: [transaction_id#726, customer_id#727, region#728, amount#729, status#730, transaction_date#731]


+--------------+------

### Exercise 2: Compare two actions

ใช้ `df_bangkok_success` จาก Exercise 1 แล้วเปรียบเทียบ action 2 แบบ:

- `.show()` ใช้ preview records กลับมาแสดงใน notebook เหมาะกับการตรวจ data sample แบบเร็ว ๆ
- `.count()` ใช้นับจำนวน rows ทั้งหมดที่ match กับ transformation plan เหมาะกับการ validate row count


In [14]:
df_bangkok_success.show()

bangkok_success_count = df_bangkok_success.count()
print("Bangkok successful transaction count:", bangkok_success_count)

# Takeaway:
# - Both can trigger Spark jobs.
# - Without cache, repeated actions may recompute the same transformation plan.

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 16, Finished, Available, Finished, False)

+--------------+-----------+-------+------+---------------+
|transaction_id|customer_id| region|amount|amount_with_fee|
+--------------+-----------+-------+------+---------------+
|          T001|        101|Bangkok|1200.5|        1236.52|
|          T002|        102|Bangkok| 250.0|          257.5|
+--------------+-----------+-------+------+---------------+

Bangkok successful transaction count: 2


### Exercise 3: Write result to a Lakehouse table

สร้าง summary จาก `df_transactions`:

- group by `status`
- count transactions
- sum amount
- write เป็น table ชื่อ `day02_status_summary`

Expected result:

- มี table `day02_status_summary` ใน Lakehouse
- read table กลับมาแล้ว `.show()` ได้

In [15]:
df_status_summary = (
    df_transactions
    .groupBy("status")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
    .orderBy("status")
)

(
    df_status_summary
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("day02_status_summary")
)

spark.read.table("day02_status_summary").show()

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 17, Finished, Available, Finished, False)

+-------+-----------------+------------+
| status|transaction_count|total_amount|
+-------+-----------------+------------+
| failed|                2|       150.0|
|pending|                1|       780.0|
|success|                5|     11430.5|
+-------+-----------------+------------+



## Common mistakes

1. **คิดว่า DataFrame transformation execute ทันทีทุกบรรทัด**

   เช่นคิดว่า `.filter()` ทำงานเสร็จแล้วทันที ทั้งที่จริง Spark แค่ build plan ไว้ก่อน

2. **ใช้ `.count()` เพื่อตรวจข้อมูลบ่อยเกินไป**

   `.count()` เป็น action ที่ต้อง scan data เพื่อรู้จำนวน row ถ้า dataset ใหญ่จะใช้ resource สูง

3. **ใช้ `.collect()` โดยไม่ระวัง**

   `collect()` ดึงข้อมูลทั้งหมดกลับมาที่ driver ถ้าข้อมูลใหญ่ driver memory อาจเต็มได้

4. **เรียก action ซ้ำโดยไม่รู้ตัว**

   ตัวอย่างเช่น `.show()`, `.count()`, `.write()` บน DataFrame เดียวกันหลายรอบ อาจทำให้ Spark compute ซ้ำ

5. **Cache ทุกอย่าง**

   Cache ช่วยได้เมื่อ DataFrame ถูกใช้ซ้ำและ compute แพง แต่ถ้า cache มากเกินไปจะกิน memory และอาจทำให้ performance แย่ลง

## Expected Output / Success Criteria

เมื่อจบ Day 02 ควรทำได้:

- อธิบายได้ว่า **Lazy Evaluation** คือ Spark จะยังไม่ execute transformation ทันที แต่จะ build plan ไว้ก่อน
- แยกได้ว่า operation ไหนคือ **Transformation** เช่น `filter`, `select`, `withColumn`, `groupBy().agg()`
- แยกได้ว่า operation ไหนคือ **Action** เช่น `show`, `count`, `collect`, `write`, `saveAsTable`
- ใช้ `.explain(mode="formatted")` เพื่อดู execution plan เบื้องต้นได้
- เข้าใจว่า `.show()`, `.count()` และ `.write` สามารถ trigger Spark job ได้
- อธิบายได้ว่า repeated actions บน DataFrame เดิมอาจทำให้ Spark compute ซ้ำ ถ้าไม่ได้ cache หรือ persist ไว้
- เข้าใจว่า `collect()` ไม่ควรใช้กับข้อมูลใหญ่ เพราะดึงข้อมูลทั้งหมดกลับมาที่ driver
- สร้าง transformation chain จาก DataFrame ได้โดยใช้ `filter`, `withColumn`, `select`
- สร้าง aggregation summary ด้วย `groupBy().agg()` ได้

## Final takeaway

Lazy Evaluation คือหนึ่งใน mindset สำคัญของ Spark:

> เราไม่ได้สั่ง Spark ให้ทำงานทีละบรรทัดแบบ pandas เสมอไป แต่เราสร้าง plan แล้ว Spark จะ execute เมื่อเจอ action

สำหรับ Data Engineer สิ่งที่ต้องจำคือ:

- ระวัง action ที่ไม่จำเป็น
- ใช้ `.explain()` เพื่อเริ่มอ่าน plan
- อย่าใช้ `.collect()` กับข้อมูลใหญ่
- Cache เฉพาะตอนมีเหตุผล
- ใน production, ทุก action คือ potential cost

## Optional cleanup

In [16]:
# spark.sql("DROP TABLE IF EXISTS day02_region_summary")
# spark.sql("DROP TABLE IF EXISTS day02_status_summary")

StatementMeta(, 2255918f-8c5f-4b09-9a78-2e4a1334395d, 18, Finished, Available, Finished, False)